# FLUXION — AI Image & Video Generator
> CTO Stack: FLUX.1-schnell (immagini) + Wan2.1-1.3B (video) + Flask REST API
> GPU: T4 16GB Kaggle | Tunnel: trycloudflare (no account)
>
## ⚡ Come usare
1. **Esegui TUTTE le celle in ordine** (una sola volta)
2. **Copia l'URL** che appare nell'ultima cella (es: `https://xxx.trycloudflare.com`)
3. **Usa l'API** dal Mac con i comandi curl nella sezione finale
>
> Il server rimane attivo per tutta la sessione Kaggle (max 12h). NON serve ricaricare le celle.

In [ ]:
# ============================================================
# CELLA 1 — INSTALL DIPENDENZE (una sola volta, ~3 min)
# ============================================================
import subprocess, sys

packages = [
    'diffusers==0.31.0',
    'transformers>=4.45.0',
    'accelerate>=1.0.0',
    'sentencepiece',
    'flask>=3.0.0',
    'flask-cors',
    'pillow>=10.0.0',
    'imageio[ffmpeg]',
    'optimum',
    'bitsandbytes',
    'safetensors',
    'opencv-python-headless',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# Installa cloudflared per il tunnel
subprocess.run([
    'wget', '-q', '-O', '/usr/local/bin/cloudflared',
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
], check=True)
subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)

print('✅ Dipendenze installate!')

In [ ]:
# ============================================================
# CELLA 2 — CARICA MODELLO IMMAGINI: FLUX.1-schnell
# (~3-5 min download, 16GB VRAM con nf4 quantization)
# ============================================================
import torch
from diffusers import FluxPipeline
from transformers import BitsAndBytesConfig

print(f'🔧 GPU: {torch.cuda.get_device_name(0)}')
print(f'🔧 VRAM totale: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB')

# Quantizzazione nf4 per FLUX in 16GB T4
nf4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

print('📥 Download FLUX.1-schnell...')
image_pipe = FluxPipeline.from_pretrained(
    'black-forest-labs/FLUX.1-schnell',
    torch_dtype=torch.bfloat16,
    quantization_config=nf4_config,
    device_map='auto'
)

print(f'✅ FLUX.1-schnell caricato!')
print(f'🔧 VRAM usata: {torch.cuda.memory_allocated(0) / 1e9:.1f}GB')

In [ ]:
# ============================================================
# CELLA 3 — CARICA MODELLO VIDEO: Wan2.1-1.3B
# (~2-3 min download, ~8GB VRAM)
# ============================================================
from diffusers import AutoPipelineForText2Video

print('📥 Download Wan2.1-1.3B...')
try:
    video_pipe = AutoPipelineForText2Video.from_pretrained(
        'Wan-AI/Wan2.1-T2V-1.3B-Diffusers',
        torch_dtype=torch.float16,
        device_map='auto'
    )
    VIDEO_AVAILABLE = True
    print('✅ Wan2.1-1.3B caricato!')
except Exception as e:
    print(f'⚠️  Video model non disponibile: {e}')
    print('   → Generazione immagini disponibile, video no.')
    VIDEO_AVAILABLE = False

print(f'🔧 VRAM totale usata: {torch.cuda.memory_allocated(0) / 1e9:.1f}GB')

In [ ]:
# ============================================================
# CELLA 4 — FLASK API SERVER
# ============================================================
import base64
import io
import time
import traceback
import threading
import imageio
import numpy as np
from flask import Flask, request, jsonify
from flask_cors import CORS
from PIL import Image

app = Flask(__name__)
CORS(app)

# ---- Lock per accesso sequenziale GPU ----
gpu_lock = threading.Lock()

@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        'status': 'ok',
        'model_image': 'FLUX.1-schnell',
        'model_video': 'Wan2.1-1.3B' if VIDEO_AVAILABLE else 'not_loaded',
        'device': str(torch.cuda.get_device_name(0)),
        'vram_used_gb': round(torch.cuda.memory_allocated(0) / 1e9, 2)
    })

@app.route('/generate', methods=['POST'])
def generate_image():
    """
    POST /generate
    Body JSON: { "prompt": "...", "width": 1024, "height": 1024, "steps": 4, "seed": 42 }
    Returns: { "image_b64": "...", "elapsed_s": 12.3 }
    """
    try:
        data = request.get_json(force=True)
        prompt = data.get('prompt', 'professional software UI screenshot, dark mode, Italian')
        width = int(data.get('width', 1024))
        height = int(data.get('height', 1024))
        steps = int(data.get('steps', 4))
        seed = int(data.get('seed', 42))

        t0 = time.time()
        with gpu_lock:
            generator = torch.Generator(device='cuda').manual_seed(seed)
            result = image_pipe(
                prompt=prompt,
                width=width,
                height=height,
                num_inference_steps=steps,
                generator=generator,
                guidance_scale=0.0  # FLUX-schnell: 0 guidance
            )
        image = result.images[0]
        elapsed = round(time.time() - t0, 2)

        # Converti in base64
        buf = io.BytesIO()
        image.save(buf, format='PNG')
        img_b64 = base64.b64encode(buf.getvalue()).decode()

        return jsonify({'image_b64': img_b64, 'elapsed_s': elapsed, 'seed': seed})

    except Exception as e:
        return jsonify({'error': str(e), 'traceback': traceback.format_exc()}), 500

@app.route('/generate-video', methods=['POST'])
def generate_video():
    """
    POST /generate-video
    Body JSON: { "prompt": "...", "frames": 16, "fps": 8, "seed": 42 }
    Returns: { "video_b64": "...", "elapsed_s": 45.2 }
    """
    if not VIDEO_AVAILABLE:
        return jsonify({'error': 'Video model not loaded'}), 503

    try:
        data = request.get_json(force=True)
        prompt = data.get('prompt', 'professional software demo, dark mode UI, smooth animation')
        frames = int(data.get('frames', 16))
        fps = int(data.get('fps', 8))
        seed = int(data.get('seed', 42))

        t0 = time.time()
        with gpu_lock:
            generator = torch.Generator(device='cuda').manual_seed(seed)
            result = video_pipe(
                prompt=prompt,
                num_frames=frames,
                generator=generator
            )
        video_frames = result.frames[0]  # lista di PIL Images
        elapsed = round(time.time() - t0, 2)

        # Converti frames in mp4 base64
        buf = io.BytesIO()
        frames_np = [np.array(f) for f in video_frames]
        writer = imageio.get_writer(buf, format='mp4', fps=fps, codec='libx264')
        for f in frames_np:
            writer.append_data(f)
        writer.close()
        video_b64 = base64.b64encode(buf.getvalue()).decode()

        return jsonify({'video_b64': video_b64, 'elapsed_s': elapsed, 'frames': frames})

    except Exception as e:
        return jsonify({'error': str(e), 'traceback': traceback.format_exc()}), 500

# Avvia Flask in thread separato (non blocca il notebook)
flask_thread = threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=5000, use_reloader=False, threaded=False),
    daemon=True
)
flask_thread.start()
print('✅ Flask server avviato su porta 5000')

In [ ]:
# ============================================================
# CELLA 5 — TUNNEL PUBBLICO (trycloudflare, NO account)
# L'URL pubblico apparirà qui sotto — COPIALO sul Mac
# ============================================================
import subprocess, re, time

# Avvia cloudflared in background
proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:5000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print('⏳ Avvio tunnel (10-15 sec)...')
public_url = None
for _ in range(60):
    line = proc.stdout.readline()
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[\w.-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            break
    time.sleep(0.5)

if public_url:
    print(f'''
╔══════════════════════════════════════════════════════╗
║  ✅ API FLUXION ATTIVA                               ║
╠══════════════════════════════════════════════════════╣
║  URL: {public_url:<46} ║
╠══════════════════════════════════════════════════════╣
║  GET  /health          → stato server                ║
║  POST /generate        → genera immagine PNG         ║
║  POST /generate-video  → genera video MP4            ║
╚══════════════════════════════════════════════════════╝

📋 COMANDI DAL MAC:

# Health check
curl {public_url}/health

# Genera immagine FLUXION (dark mode UI)
curl -s -X POST {public_url}/generate \\
  -H "Content-Type: application/json" \\
  -d '{{"prompt":"FLUXION gestionale italiano dark mode, calendário appuntamenti, teal accent, professional UI screenshot 2026","width":1280,"height":720,"steps":4,"seed":42}}' \\
  | python3 -c "import sys,json,base64; d=json.load(sys.stdin); open('fluxion-ui.png','wb').write(base64.b64decode(d['image_b64'])); print(f\"✅ Salvata! Tempo: {{d['elapsed_s']}}s\")"

# Genera video demo FLUXION
curl -s -X POST {public_url}/generate-video \\
  -H "Content-Type: application/json" \\
  -d '{{"prompt":"FLUXION Italian PMI software demo, dark mode UI, smooth calendar animation, teal accent","frames":16,"fps":8,"seed":42}}' \\
  | python3 -c "import sys,json,base64; d=json.load(sys.stdin); open('fluxion-demo.mp4','wb').write(base64.b64decode(d['video_b64'])); print(f\"✅ Video salvato! Tempo: {{d['elapsed_s']}}s\")"
''')
else:
    print('❌ Tunnel non avviato — riprova la cella 5')

## 🎨 Prompt Templates FLUXION

Copia questi prompt nell'endpoint `/generate` per screenshot FLUXION photorealistici:

```
# Calendario prenotazioni
FLUXION Italian salon booking software, dark mode UI, weekly calendar view, teal-500 accent, 
appointment cards with client names, professional desktop app 2026, glassmorphism cards

# Dashboard principale
FLUXION PMI management software dashboard, dark slate-900 background, KPI cards with revenue stats,
teal accent colors, Italian language labels, sidebar navigation, modern desktop app UI 2026

# Scheda cliente fitness
FLUXION gym client profile card, dark mode, BMI gauge circular indicator, fitness metrics charts,
teal progress bars, Italian UI labels, professional sports management software 2026

# Voice agent Sara
FLUXION voice AI assistant Sara interface, dark mode, waveform animation, Italian booking dialog,
teal accent, modern desktop app, conversation UI with appointment confirmation

# Fatturazione SDI
FLUXION Italian invoicing software, dark mode, FatturaPA XML view, SDI status badges,
teal accent, professional accounting UI 2026
```